In [1]:
#
# save_spectrogram_vector.py
#
# Reformat spectraogram data into training-ready .csv files
#
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
#

#%% Imports

import os
from fnmatch import fnmatch
import cv2
import numpy as np
import random
import traceback

In [2]:
#%% Path configuration

current_dir = os.path.abspath("D:\\beluga")
data_dir = os.path.join(current_dir, "Data")

model_dir = os.path.join(current_dir,"Model")
spectrogram_dir = os.path.join(data_dir,"Extracted_Spectrogram")
output_spectrogram_vector_dir = os.path.join(data_dir, "Output_Spectrogram_Vector")

for i in [model_dir, output_spectrogram_vector_dir]:

    if not os.path.exists(i):
        os.makedirs(i)


#%% Load spectrograms


"""
Because we have a severe class imbalance, I am looking to increase the number of certain classes in the training set.
During the spectrogram generation process, the species and data-source were included with the filename. 
For example, beluga from sub-1khz dataset (bs1k) and false beluga from the review of the 237 dataset (fml237).
Given that we can identify the datasource, we will first create a vector for each distinct class, 
then combine data from these vectors as necessary. 

"""
spectrograms_BS1K = []
spectrograms_BDB = []
spectrograms_FML237 = []
spectrograms_FS1K = []
spectrograms_FDB = []
spectrograms_N = []

filenames_BS1K = []
filenames_BDB = []
filenames_FML237 = []
filenames_FS1K = []
filenames_FDB = [] 
filenames_N = []

ncol, nrow = 300, 300



In [15]:
index = 0
for path, subdirs, files in os.walk(spectrogram_dir):
    for filename in files:
        if(index % 5000 == 0):
            print('At ', index)
        try:

            if fnmatch(filename, '*B-DB.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                
                spectrograms_BDB.append(img)
                filenames_BDB.append(path + '/' + filename)
            
            if fnmatch(filename, '*B-S1K.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                
                spectrograms_BS1K.append(img)
                filenames_BS1K.append(path + '/' + filename)
        
            if fnmatch(filename, '*F-ML237.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                
                spectrograms_FML237.append(img)
                filenames_FML237.append(path + '/' + filename)
            
            if fnmatch(filename, '*F-S1K.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                
                spectrograms_FS1K.append(img)
                filenames_FS1K.append(path + '/' + filename)
            
                
            if fnmatch(filename, '*F-DB.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                spectrograms_FDB.append(img)
                filenames_FDB.append(path + '/' + filename)
         

            if fnmatch(filename, '*_N.png'):
                img = cv2.imread(path + '/' + filename)
                img = cv2.resize(img, (ncol, nrow))
                spectrograms_N.append(img)
                filenames_N.append(path + '/' + filename)
    
        except:
            traceback.print_exc()
        index += 1

print("Sub 1Khz beluga", len(spectrograms_BS1K))
print("DB beluga", len(spectrograms_BDB))
print("False ML beluga", len(spectrograms_FML237))
print("False sub 1Khz beluga", len(spectrograms_FS1K))
print("False DB beluga", len(spectrograms_FDB))
print("Non-beluga", len(spectrograms_N))

At  0
At  5000
At  10000
At  15000
At  20000
At  25000
At  30000
At  35000
At  40000
At  45000
At  50000
At  55000
At  60000
At  65000
At  70000
At  75000
At  80000
Sub 1Khz beluga 1226
DB beluga 50761
False ML beluga 267
False sub 1Khz beluga 130
False DB beluga 16762
Non-beluga 69145


# we need to generate a 70/30 train and validate set

To do this, we first make a subset of each list based on 70/30 split randomly

we then get the difference between the full filename list and subsample by converting the former into a set and substracting the latter. Then we cast back to lists. 

In [16]:
def split_dataset(input_list, input_filenames, proportion):
    
        length_of_list = len(input_list)
        sample_size = int(proportion * length_of_list)
        #print(length_of_list, type(length_of_list), proportion, type(proportion))
        
        
        val_sample_index = sorted(random.sample(range(length_of_list), sample_size))
        
        #print(length_of_list, sample_size, val_sample_index)
        
        val_spectrogram_sample = [input_list[i] for i in val_sample_index]

        val_filename_sample = [input_filenames[i] for i in val_sample_index]
        
        return val_spectrogram_sample, val_filename_sample, val_sample_index
split_dataset([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], 0.3)

([4, 5, 11, 14, 16, 18], [4, 5, 11, 14, 16, 18], [3, 4, 10, 13, 15, 17])

In [17]:
traintest_validation_proportion = 0.3

split_bs1k = split_dataset(spectrograms_BS1K, filenames_BS1K, traintest_validation_proportion)
split_bdb = split_dataset(spectrograms_BDB, filenames_BDB, traintest_validation_proportion)

split_fml237 = split_dataset(spectrograms_FML237, filenames_FML237, traintest_validation_proportion)
split_fs1k = split_dataset(spectrograms_FS1K, filenames_FS1K, traintest_validation_proportion)
split_fdb = split_dataset(spectrograms_FDB, filenames_FDB, traintest_validation_proportion)
split_n = split_dataset(spectrograms_N, filenames_N, traintest_validation_proportion)


val_spectrograms_BS1K = split_bs1k[0]
val_spectrograms_BDB = split_bdb[0]
val_spectrograms_FML237 = split_fml237[0]
val_spectrograms_FS1K = split_fs1k[0]
val_spectrograms_FDB = split_fdb[0]
val_spectrograms_N = split_n[0]

val_filenames_BS1K = split_bs1k[1]
val_filenames_BDB = split_bdb[1]
val_filenames_FML237 = split_fml237[1]
val_filenames_FS1K = split_fs1k[1]
val_filenames_FDB = split_fdb[1] 
val_filenames_N = split_n[1]

index_val_filenames_BS1K = split_bs1k[2]
index_val_filenames_BDB = split_bdb[2]
index_val_filenames_FML237 = split_fml237[2]
index_val_filenames_FS1K = split_fs1k[2]
index_val_filenames_FDB = split_fdb[2] 
index_val_filenames_N = split_n[2]


## we now have our validation split: 30% of each class
But, we need to remove these validation detections from the filename list as well as the spectrogram list. We do this with a list comprehension

In [18]:

test_list = ['a', 'b', 'c', 'd', 'e', 'f'] 
test_index = [1,5]

#remove index 1, 5 (b and f from the list)
#expect output is a, c, d, e
[i for j, i in enumerate(test_list) if j not in test_index]

['a', 'c', 'd', 'e']

In [19]:
print('Before Partitioning')
print("Sub 1Khz beluga", len(spectrograms_BS1K))
print("DB beluga", len(spectrograms_BDB))
print("False ML beluga", len(spectrograms_FML237))
print("False sub 1Khz beluga", len(spectrograms_FS1K))
print("False DB beluga", len(spectrograms_FDB))
print("Non-beluga", len(spectrograms_N))
print('\n')
print('Subset')
print("Validation Sub 1Khz beluga", len(val_spectrograms_BS1K))
print("Validation DB beluga", len(val_spectrograms_BDB))
print("Validation False ML beluga", len(val_spectrograms_FML237))
print("Validation False sub 1Khz beluga", len(val_spectrograms_FS1K))
print("Validation False DB beluga", len(val_spectrograms_FDB))
print("Validation Non-beluga", len(val_spectrograms_N))




spectrograms_BS1K = [i for j, i in enumerate(spectrograms_BS1K) if j not in index_val_filenames_BS1K]
spectrograms_BDB = [i for j, i in enumerate(spectrograms_BDB) if j not in index_val_filenames_BDB]
spectrograms_FML237 = [i for j, i in enumerate(spectrograms_FML237) if j not in index_val_filenames_FML237]
spectrograms_FS1K = [i for j, i in enumerate(spectrograms_FS1K) if j not in index_val_filenames_FS1K]
spectrograms_FDB = [i for j, i in enumerate(spectrograms_FDB) if j not in index_val_filenames_FDB]
spectrograms_N = [i for j, i in enumerate(spectrograms_N) if j not in index_val_filenames_N]

filenames_BS1K = [i for j, i in enumerate(filenames_BS1K) if j not in index_val_filenames_BS1K]
filenames_BDB = [i for j, i in enumerate(filenames_BDB) if j not in index_val_filenames_BDB]
filenames_FML237 = [i for j, i in enumerate(filenames_FML237) if j not in index_val_filenames_FML237]
filenames_FS1K = [i for j, i in enumerate(filenames_FS1K) if j not in index_val_filenames_FS1K]
filenames_FDB  = [i for j, i in enumerate(filenames_FDB) if j not in index_val_filenames_FDB]
filenames_N = [i for j, i in enumerate(filenames_N) if j not in index_val_filenames_N]



Before Partitioning
Sub 1Khz beluga 1226
DB beluga 50761
False ML beluga 267
False sub 1Khz beluga 130
False DB beluga 16762
Non-beluga 69145


Subset
Validation Sub 1Khz beluga 367
Validation DB beluga 15228
Validation False ML beluga 80
Validation False sub 1Khz beluga 39
Validation False DB beluga 5028
Validation Non-beluga 20743


In [20]:
print("Validation Sub 1Khz beluga", len(val_spectrograms_BS1K), len(val_filenames_BS1K))
print("Validation DB beluga", len(val_spectrograms_BDB), len(val_filenames_BDB))
print("Validation False ML beluga", len(val_spectrograms_FML237), len(val_filenames_FML237))
print("Validation False sub 1Khz beluga", len(val_spectrograms_FS1K), len(val_filenames_FS1K))
print("Validation False DB beluga", len(val_spectrograms_FDB), len(val_filenames_FDB))
print("Validation Non-beluga", len(val_spectrograms_N), len(val_filenames_N))

print('New training set size')
print("Train-Test Sub 1Khz beluga", len(spectrograms_BS1K), len(filenames_BS1K))
print("Train-Test DB beluga", len(spectrograms_BDB), len(filenames_BDB))
print("Train-Test False ML beluga", len(spectrograms_FML237), len(filenames_FML237))
print("Train-Test False sub 1Khz beluga", len(spectrograms_FS1K), len(filenames_FS1K))
print("Train-Test False DB beluga", len(spectrograms_FDB), len(filenames_FDB))
print("Train-Test Non-beluga", len(spectrograms_N), len(filenames_N))


Validation Sub 1Khz beluga 367 367
Validation DB beluga 15228 15228
Validation False ML beluga 80 80
Validation False sub 1Khz beluga 39 39
Validation False DB beluga 5028 5028
Validation Non-beluga 20743 20743
New training set size
Train-Test Sub 1Khz beluga 859 859
Train-Test DB beluga 35533 35533
Train-Test False ML beluga 187 187
Train-Test False sub 1Khz beluga 91 91
Train-Test False DB beluga 11734 11734
Train-Test Non-beluga 48402 48402


[1]

# Save Validation Set 

In [21]:


"""
Vectors to np array file
"""
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_BS1K_300_300"), np.asarray(val_spectrograms_BS1K))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_BDB_300_300"), np.asarray(val_spectrograms_BDB))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_FML237_300_300"), np.asarray(val_spectrograms_FML237))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_FS1K_300_300"), np.asarray(val_spectrograms_FS1K))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_FDB_300_300"), np.asarray(val_spectrograms_FDB))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"spectrograms_N_300_300"), np.asarray(val_spectrograms_N))




"""
File lists to CSV
"""

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_BS1K.csv"), "w") as f:
    for filename in val_filenames_BS1K:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_BDB.csv"), "w") as f:
    for filename in val_filenames_BDB:
        f.write(filename)
        f.write('\n')


with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_FML237.csv"), "w") as f:
    for filename in val_filenames_FML237:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_FS1K.csv"), "w") as f:
    for filename in val_filenames_FS1K:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_FDB.csv"), "w") as f:
    for filename in val_filenames_FDB:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "validation_"+"filenames_N.csv"), "w") as f:
    for filename in val_filenames_N:
        f.write(filename)
        f.write('\n')

# Save Training Set

In [22]:
#spectrograms_BS1K = np.asarray(spectrograms_BS1K)
#spectrograms_BDB = np.asarray(spectrograms_BDB)
#spectrograms_FML237 = np.asarray(spectrograms_FML237)
#spectrograms_FS1K = np.asarray(spectrograms_FS1K)
#spectrograms_FDB = np.asarray(spectrograms_FDB)
#spectrograms_N = np.asarray(spectrograms_N)


"""
Vectors to np array file
"""
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_BS1K_300_300"), np.asarray(spectrograms_BS1K))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_BDB_300_300"), np.asarray(spectrograms_BDB))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_FML237_300_300"), np.asarray(spectrograms_FML237))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_FS1K_300_300"), np.asarray(spectrograms_FS1K))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_FDB_300_300"), np.asarray(spectrograms_FDB))
np.save(os.path.join(data_dir, output_spectrogram_vector_dir, "spectrograms_N_300_300"), np.asarray(spectrograms_N))




"""
File lists to CSV
"""

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_BS1K.csv"), "w") as f:
    for filename in filenames_BS1K:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_BDB.csv"), "w") as f:
    for filename in filenames_BDB:
        f.write(filename)
        f.write('\n')


with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_FML237.csv"), "w") as f:
    for filename in filenames_FML237:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_FS1K.csv"), "w") as f:
    for filename in filenames_FS1K:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_FDB.csv"), "w") as f:
    for filename in filenames_FDB:
        f.write(filename)
        f.write('\n')

with open(os.path.join(data_dir, output_spectrogram_vector_dir, "filenames_N.csv"), "w") as f:
    for filename in filenames_N:
        f.write(filename)
        f.write('\n')


# Blending data to create training sets
We now can modify the "mix" of spectrograms that go into the training sample. This capability is required due to the class imbalances between normal detections (i.e. from the CIBA2 database), sub-1KhZ detections, and false ML detections. Because there are fewer of these exemplars, we can utilize methods [1] to oversample the class or to utilize some stratified-sampling method. Our ultimate goal is to develop one sample vector for beluga, false beluga, and none (B, F, N).

To keep track of these different approaches, this notebook will be utlized to generate training sets and to track their compositions.

[1] https://machinelearningmastery.com/tactics-to-combat-imbalanced-classes-in-your-machine-learning-dataset/


In [23]:
#Test data to evaluate logic
#We prepare some dummy data because the actual files are way too abstract to comprehend.
sub1k_beluga = ["alice", "bob", "charlie","donny", "ellen"]
sub1k_beluga_filenames = ['a','b','c','d','e']

normal_beluga =["fred", "george", "helen","ian", "jackie"]
normal_beluga_filenames = ['f','g','h','i','j']







A package is a list containing:
1. The name of the dataset (e.g. no animals present, false beluga, etc.) (string)
2. A python list of spectrograms
3. The accompanying filenames for those spectrograms (list)
4. The number of samples to pull from this dataset (int)
5. The number of times to include the spectrogram in the spectrogram training set (int)

This data 

We include the dataset name as a string because numpy arrays and python list objects are not named. Given the need to pontentially vary sample sizes and number of times to include a particular type of dataset into a given training set, we need to devise a method for programtically creating training sets and tracking how they are construcuted. Developing an algorithm to complete this work offers two key benefits. First, it allows for rapidly changing training dataset composition and reduces the number of changes that need to be made to the script when making modifications. 

In [24]:

def blend_training_dataset(variant_name, packages):
    print("processing {variant}".format(variant=variant_name))
    output_file_list = []  
    output_spectrograms = []
    training_set_name_list = [variant_name]
    
    for job in packages:
        #print("Job: ", job)

        job_name = job[0]
        spectrograms_list = job[1]   
        filenames = job[2]
        
        sample_size = job[3]
        multiplier = job[4]
        
        training_set_name = "-".join([job_name,"n"+str(sample_size),"x"+str(multiplier)])
        training_set_name_list.append(training_set_name)
        
        print("choosing {sample_size} random values and adding to training set {multiplier} times".format(sample_size = sample_size,
                                                                                                             multiplier = multiplier))
        sample_index = sorted(random.sample(range(len(spectrograms_list)), sample_size))
        
        spectrogram_sample = [spectrograms_list[i] for i in sample_index]

        filename_sample = [filenames[i] for i in sample_index]
        
        #print(spectrogram_sample, filename_sample)
        
        #add samples to outputs the number of specified times
        for i in range(0,multiplier):
            #print("appending sample", spectrogram_sample)
            output_spectrograms = output_spectrograms + spectrogram_sample

            #output_spectrogram_vector = output_spectrogram_vector + spectrogram_sample
            #output_spectrogram_vector = np.concatenate((output_spectrogram_vector, spectrogram_sample),axis=None)
            output_file_list = output_file_list + filename_sample

    
    training_set_name = "_".join(training_set_name_list)
    
    
    #finally we convert the python image list into an array
    output_spectrogram_vector = np.asarray(output_spectrograms)
    print("length of spectrogram list: ", len(spectrograms_list))
    print("shape of output spectrogram array: ", output_spectrogram_vector.shape)
    print("length of filename list: ", len(output_file_list))
    #print("Final training spectrogram set ",training_set_name, output_spectrogram_vector)
    spectrogram_file_path = os.path.join(output_spectrogram_vector_dir,"-".join([training_set_name,'spectrograms_sample_300_300.npy']))
    
    np.save(spectrogram_file_path, output_spectrogram_vector)
    print("saved ", spectrogram_file_path)
    
    
    csv_file_path = os.path.join(output_spectrogram_vector_dir,"-".join([training_set_name,'filenames_sample.csv']))
                                 
    with open(csv_file_path,'w') as f:
        for filename in output_file_list:
            f.write(filename)
            f.write('\n')
    print("saved CSV file", csv_file_path)
    
    print("spectrogram vector length: ",len(output_spectrogram_vector))
    print("filename length: ", len(output_file_list))

    return training_set_name, output_spectrogram_vector, output_file_list




In [25]:
#first we create package (python list) with the aforementioned objects.  
sub1k_training_pkg = "sub1k-beluga", sub1k_beluga, sub1k_beluga_filenames, 3, 2
normal_beluga_pkg = "normal-beluga", normal_beluga, normal_beluga_filenames, 3, 1

#we then need to create a list of packages that are together
#e.g. all positives, all negatives, all false
beluga_positive_pkgs = [sub1k_training_pkg, normal_beluga_pkg]
beluga_false_pkgs = []
beluga_negative_pkgs = []

#then we run the code
test = blend_training_dataset("TEST", beluga_positive_pkgs)
#print("output:", test[1], test[2])

processing TEST
choosing 3 random values and adding to training set 2 times
choosing 3 random values and adding to training set 1 times
length of spectrogram list:  5
shape of output spectrogram array:  (9,)
length of filename list:  9
saved  D:\beluga\Data\Output_Spectrogram_Vector\TEST_sub1k-beluga-n3-x2_normal-beluga-n3-x1-spectrograms_sample_300_300.npy
saved CSV file D:\beluga\Data\Output_Spectrogram_Vector\TEST_sub1k-beluga-n3-x2_normal-beluga-n3-x1-filenames_sample.csv
spectrogram vector length:  9
filename length:  9


['c', 'd', 'e', 'c', 'd', 'e', 'g', 'i', 'j']

In [23]:
test[1]

array(['fred', 'george', 'helen', 'ian', 'jackie'], dtype='<U6')

If we find this satisfactory, we can proceed with actually processing beluga training data

## As a reminder, we previously randomly partitioned each class in a 70-30 train-test split

# Training Set 1: Use Full Dataset, Only Once

In [26]:
#first we create package (python list) with the aforementioned objects.  
#format is dataset name, spectogram array, filename list, sample size, multiplier (how many times to include the sample)
beluga_sub1khz_training_pkg = "BS1KHZ", spectrograms_BS1K, filenames_BS1K, len(spectrograms_BS1K), 1
beluga_db_pkg = "BDB", spectrograms_BDB, filenames_BDB, len(spectrograms_BDB), 1

false_ml237_pkg = "FML237", spectrograms_FML237, filenames_FML237, len(spectrograms_FML237), 1
false_sub1khz_pkg = "FS1KHZ", spectrograms_FS1K, filenames_FS1K, len(spectrograms_FS1K), 1
false_db_pkg = "FDB", spectrograms_FDB, filenames_FDB,len(spectrograms_FDB), 1

no_beluga_pkg = "N", spectrograms_N, filenames_N, len(spectrograms_N), 1

"""
We then need to create a list of packages that are associated (e.g. all positives, all negatives, all false)
Should we wish to add or subtract a dataset from training, we can do so here. 
"""
beluga_positive_pkgs = [beluga_sub1khz_training_pkg, beluga_db_pkg]
beluga_false_pkgs = [false_ml237_pkg,false_sub1khz_pkg,false_db_pkg]
beluga_negative_pkgs = [no_beluga_pkg]

#then we run the code
blend_training_dataset("beluga-positive", beluga_positive_pkgs)
blend_training_dataset("beluga-false", beluga_false_pkgs)
beluga_negative = blend_training_dataset("beluga-negative", beluga_negative_pkgs)

processing beluga-positive
choosing 859 random values and adding to training set 1 times
choosing 35533 random values and adding to training set 1 times
length of spectrogram list:  35533
shape of output spectrogram array:  (36392, 300, 300, 3)
length of filename list:  36392
saved  D:\beluga\Data\Output_Spectrogram_Vector\beluga-positive_BS1KHZ-n859-x1_BDB-n35533-x1-spectrograms_sample_300_300.npy
saved CSV file D:\beluga\Data\Output_Spectrogram_Vector\beluga-positive_BS1KHZ-n859-x1_BDB-n35533-x1-filenames_sample.csv
spectrogram vector length:  36392
filename length:  36392
processing beluga-false
choosing 187 random values and adding to training set 1 times
choosing 91 random values and adding to training set 1 times
choosing 11734 random values and adding to training set 1 times
length of spectrogram list:  11734
shape of output spectrogram array:  (12012, 300, 300, 3)
length of filename list:  12012
saved  D:\beluga\Data\Output_Spectrogram_Vector\beluga-false_FML237-n187-x1_FS1KHZ-

In [ ]:
output_spectrogram_vector_dir

## Original Code for Generating a Random Sample

#%% Generate a random sample of spectrograms for training
        
random.seed(40)
"""
#pick the indicies
spectrograms_B_sample_index = sorted(random.sample(range(len(filenames_B)), 50000))
spectrograms_F_sample_index = sorted(random.sample(range(len(filenames_F)), 100000))
spectrograms_N_sample_index = sorted(random.sample(range(len(filenames_N)), 20000))

#grap the specified spectrograms at that index
spectrograms_B_sample = spectrograms_B[spectrograms_B_sample_index]
spectrograms_F_sample = spectrograms_F[spectrograms_F_sample_index]
spectrograms_N_sample = spectrograms_N[spectrograms_N_sample_index]
        
#get 
filenames_B_sample = np.asarray(filenames_B)[spectrograms_B_sample_index].tolist()
filenames_F_sample = np.asarray(filenames_F)[spectrograms_F_sample_index].tolist()
filenames_N_sample = np.asarray(filenames_N)[spectrograms_N_sample_index].tolist()

np.save(data_dir + output_spectrogram_vector_dir + "spectrograms_B_sample_300_300", spectrograms_B_sample)
np.save(data_dir + output_spectrogram_vector_dir + "spectrograms_F_sample_300_300", spectrograms_F_sample)
np.save(data_dir + output_spectrogram_vector_dir + "spectrograms_N_sample_300_300", spectrograms_N_sample)
"""
with open(data_dir + output_spectrogram_vector_dir + "filenames_B_sample.csv",'w') as f:
    for filename in filenames_B_sample:
        f.write(filename)
        f.write('\n')

with open(data_dir + output_spectrogram_vector_dir + "filenames_F_sample.csv",'w') as f:
    for filename in filenames_F_sample:
        f.write(filename)
        f.write('\n')

with open(data_dir + output_spectrogram_vector_dir + "filenames_N_sample.csv",'w') as f:
    for filename in filenames_N_sample:
        f.write(filename)
        f.write('\n')